# Most Important Model Results

Summary of the main findings from the soft-label reappraisal classification experiments.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / 'data' / 'annotate-lime.csv').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate the project root containing data/annotate-lime.csv.')
RESULTS_DIR = PROJECT_DIR / 'Code' / 'BERT_models' / 'experiments' / 'baseline'
TUNING_DIR = PROJECT_DIR / 'Code' / 'BERT_models' / 'experiments' / 'lr_epoch_tuning'

MPL_CACHE_DIR = PROJECT_DIR / '.matplotlib_cache'
MPL_CACHE_DIR.mkdir(exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE_DIR))

MODEL_DISPLAY_NAMES = {
    'bert_base_uncased': 'BERT-base',
    'bert_large_uncased': 'BERT-large',
    'modernbert_base': 'ModernBERT-base',
    'roberta_base': 'RoBERTa-base',
    'roberta_large': 'RoBERTa-large',
}

MODEL_COLORS = {
    'BERT-base': '#3b82f6',
    'BERT-large': '#0f766e',
    'ModernBERT-base': '#f59e0b',
    'RoBERTa-base': '#8b5cf6',
    'RoBERTa-large': '#dc2626',
}

plt.rcParams.update({
    'figure.figsize': (8.5, 4.8),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.22,
    'font.size': 10.5,
})

print('Results directory:', RESULTS_DIR)
print('Tuning directory:', TUNING_DIR)


In [ ]:
def base_experiment_name(experiment_name: str) -> str:
    return experiment_name.split('_lr_')[0]


def load_metrics_row(metrics_path: Path, source: str) -> dict:
    payload = json.loads(metrics_path.read_text())
    config = payload.get('config', {})
    experiment_name = payload['experiment_name']
    base_name = base_experiment_name(experiment_name)
    soft = payload['soft_summary']
    binary = payload['binary_scheme_metrics']

    return {
        'source': source,
        'model': MODEL_DISPLAY_NAMES.get(base_name, base_name),
        'experiment_name': experiment_name,
        'model_name': payload.get('model_name'),
        'seed': payload['run_id'].replace('random_seed', ''),
        'learning_rate': config.get('learning_rate', 'original'),
        'scheduler': config.get('scheduler_type', 'linear'),
        'best_epoch': payload['best_epoch'],
        'validation_loss': payload['best_val_loss'],
        'macro_mae': soft['macro_mae'],
        'macro_rmse': soft['macro_rmse'],
        'mean_pearson': soft['mean_pearson'],
        'mean_spearman': soft['mean_spearman'],
        'majority_macro_f1': binary['majority']['tuned_binary_metrics']['macro_f1'],
        'strong_presence_macro_f1': binary['strong_presence']['tuned_binary_metrics']['macro_f1'],
        'weak_presence_macro_f1': binary['weak_presence']['tuned_binary_metrics']['macro_f1'],
    }


rows = []
for path in sorted(RESULTS_DIR.glob('*/*/*_random_seed*_metrics.json')):
    rows.append(load_metrics_row(path, 'Original runner'))
for path in sorted(TUNING_DIR.glob('*_metrics.json')):
    rows.append(load_metrics_row(path, 'LR/epoch tuning'))

results = pd.DataFrame(rows)
results['learning_rate_label'] = results['learning_rate'].map(
    lambda value: f'{float(value):.0e}' if isinstance(value, (int, float, np.floating)) else str(value)
)
results['condition'] = (
    results['model']
    + ' | lr=' + results['learning_rate_label']
    + ' | ' + results['scheduler']
    + ' | seed=' + results['seed'].astype(str)
)

key_results = (
    results
    .sort_values(['macro_mae', 'validation_loss'], ascending=[True, True])
    .reset_index(drop=True)
)
key_results.insert(0, 'rank', range(1, len(key_results) + 1))

TUNING_DIR.mkdir(exist_ok=True)
key_results.to_csv(TUNING_DIR / 'important_key_results_table.csv', index=False)

display(
    key_results[[
        'rank', 'source', 'model', 'seed', 'learning_rate_label', 'scheduler',
        'best_epoch', 'validation_loss', 'macro_mae', 'mean_pearson',
        'majority_macro_f1', 'strong_presence_macro_f1',
    ]]
    .head(12)
    .round(4)
)


### Runs

RoBERTa-large currently has the strongest single run. BERT-large has been validated more.


In [ ]:
top_runs = key_results.head(8).copy()
plot_labels = top_runs['condition']

fig, ax = plt.subplots(figsize=(10, 5.2))
colors = [MODEL_COLORS.get(model, '#6b7280') for model in top_runs['model']]
bars = ax.barh(plot_labels[::-1], top_runs['macro_mae'][::-1], color=colors[::-1])
ax.set_title('Top Individual Runs by Macro MAE')
ax.set_xlabel('Macro MAE lower is better')
ax.set_ylabel('')

for bar in bars:
    value = bar.get_width()
    ax.text(value + 0.001, bar.get_y() + bar.get_height() / 2, f'{value:.3f}', va='center')

plt.tight_layout()
plt.show()


## Training Curves For Top Runs

These curves show where early stopping selected the best checkpoint for the leading runs.


In [ ]:
def parse_history_filename(history_path: Path) -> dict:
    name = history_path.name.replace('_training_history.csv', '')
    experiment_name, seed_text = name.rsplit('_random_seed', 1)
    base_name = base_experiment_name(experiment_name)
    lr_label = 'original'
    scheduler = 'linear'
    if '_lr_' in experiment_name:
        lr_part = experiment_name.split('_lr_', 1)[1]
        if '_sched_' in lr_part:
            lr_label, scheduler_part = lr_part.split('_sched_', 1)
            scheduler = scheduler_part.replace('-', '_')
        else:
            lr_label = lr_part
        lr_label = lr_label.replace('_', '-')
    return {
        'experiment_name': experiment_name,
        'model': MODEL_DISPLAY_NAMES.get(base_name, base_name),
        'seed': seed_text,
        'learning_rate_label': lr_label,
        'scheduler': scheduler,
    }

history_rows = []
for history_path in sorted(TUNING_DIR.glob('*_training_history.csv')):
    meta = parse_history_filename(history_path)
    history = pd.read_csv(history_path)
    for key, value in meta.items():
        history[key] = value
    history['condition'] = (
        history['model']
        + ' | lr=' + history['learning_rate_label']
        + ' | ' + history['scheduler']
        + ' | seed=' + history['seed'].astype(str)
    )
    history_rows.append(history)

histories = pd.concat(history_rows, ignore_index=True) if history_rows else pd.DataFrame()

curve_conditions = set(comparison.head(6)['condition'])
curve_df = histories[histories['condition'].isin(curve_conditions)].copy()

fig, ax = plt.subplots(figsize=(10, 5.3))
for condition, history in curve_df.groupby('condition'):
    history = history.sort_values('epoch')
    model = history['model'].iloc[0]
    color = MODEL_COLORS.get(model, '#6b7280')
    ax.plot(history['epoch'], history['val_loss'], marker='o', linewidth=2, label=condition, color=color, alpha=0.82)
    best_idx = history['val_loss'].idxmin()
    ax.scatter(history.loc[best_idx, 'epoch'], history.loc[best_idx, 'val_loss'], color=color, s=60, zorder=3)

ax.set_title('Validation Loss Curves For Leading Tuned Runs')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation loss')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
